In [12]:
%pip -q install "qiskit==2.2.3" "numpy==2.3.5" "scipy==1.16.3"

In [13]:
# QAOA 작동 원리
# 1. 의사결정을 이진 변수로 표현하기
import numpy as np
n = 2
bits = ((np.arange(2**n)[:, None] >> np.arange(n)) & 1).astype(int)
print('변수 순서 [x0,x1]:',bits.tolist())
assert bits.tolist()==[[0,0],[1,0],[0,1],[1,1]]

변수 순서 [x0,x1]: [[0, 0], [1, 0], [0, 1], [1, 1]]


In [14]:
# 2. 제약조건을 비용으로 표현하기
import numpy as np
n = 2
bits = ((np.arange(2**n)[:, None] >> np.arange(n)) & 1).astype(int)
energy = np.array([(x[0]+x[1]-1)**2 for x in bits],dtype=float)
print('00,01,10,11의 비용:',energy)
assert np.array_equal(energy,[1,0,0,1])

00,01,10,11의 비용: [1. 0. 0. 1.]


In [15]:
import numpy as np
n = 2
bits = ((np.arange(2**n)[:, None] >> np.arange(n)) & 1).astype(int)
cost = np.array([8.,13.])  # 후보별 비용(만원)
original_cost = bits @ cost
penalty = 100.0
selected_count = bits.sum(axis=1)
target = 1
penalized = original_cost + penalty*(selected_count-target)**2
print('원래 비용:',original_cost)
print('선택 개수:',selected_count)
print('벌점 포함 비용:',penalized)
assert np.array_equal(penalized,[100,8,13,121])

원래 비용: [ 0.  8. 13. 21.]
선택 개수: [0 1 1 2]
벌점 포함 비용: [100.   8.  13. 121.]


In [16]:
# 이진 변수와 Pauli Z의 연결
import numpy as np
n = 2
bits = ((np.arange(2**n)[:, None] >> np.arange(n)) & 1).astype(int)
z = 1-2*bits
assert np.array_equal((1-z)//2,bits)
print('Z 고윳값:',z.tolist())
print('복원한 선택 변수:',((1-z)//2).tolist())

Z 고윳값: [[1, 1], [-1, 1], [1, -1], [-1, -1]]
복원한 선택 변수: [[0, 0], [1, 0], [0, 1], [1, 1]]


In [17]:
import numpy as np
n = 2
bits = ((np.arange(2**n)[:, None] >> np.arange(n)) & 1).astype(int)
z = 1-2*bits
energy = (bits.sum(axis=1)-1)**2
ising_energy = 0.5+0.5*z[:,0]*z[:,1]
print('QUBO 에너지:',energy)
print('Ising 에너지:',ising_energy)
assert np.allclose(ising_energy,energy)

QUBO 에너지: [1 0 0 1]
Ising 에너지: [1. 0. 0. 1.]


In [19]:
# 일반 QUBO를 Ising 계수로 전개하기
import numpy as np
n = 2
bits = ((np.arange(2**n)[:, None] >> np.arange(n)) & 1).astype(int)
constant = 1.0
a = np.array([-1.0,-1.0])
b = {(0,1):2.0}
energy = constant+bits@a
for (i,j),value in b.items():
    energy += value*bits[:,i]*bits[:,j]
print('QUBO 에너지:',energy)
assert np.allclose(energy,[1,0,0,1])

QUBO 에너지: [1. 0. 0. 1.]


In [20]:
import numpy as np
n = 2
bits = ((np.arange(2**n)[:, None] >> np.arange(n)) & 1).astype(int)
constant = 1.0
a = np.array([-1.0,-1.0])
b = {(0,1):2.0}
h = -a/2
offset = constant+a.sum()/2+sum(b.values())/4
zz = {key:value/4 for key,value in b.items()}
for (i,j),value in b.items():
    h[i] -= value/4
    h[j] -= value/4
z = 1-2*bits
ising_energy = offset+z@h
for (i,j),value in zz.items():
    ising_energy += value*z[:,i]*z[:,j]
print('상수:',offset,'Z 계수:',h,'ZZ 계수:',zz)
print('Ising 에너지:',ising_energy)
assert np.allclose(ising_energy,[1,0,0,1])

상수: 0.5 Z 계수: [0. 0.] ZZ 계수: {(0, 1): 0.5}
Ising 에너지: [1. 0. 0. 1.]


In [21]:
# 초기 중첩과 진폭
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
qc = QuantumCircuit(2)
qc.h([0,1])
state = Statevector.from_instruction(qc)
amplitude = state.data
probability = state.probabilities()
print('진폭:',amplitude)
print('확률:',probability)
assert np.allclose(probability,[0.25]*4)

진폭: [0.5+0.j 0.5+0.j 0.5+0.j 0.5+0.j]
확률: [0.25 0.25 0.25 0.25]


In [22]:
# 비용 회로가 위상을 바꾸는 과정
import numpy as np
n = 2
bits = ((np.arange(2**n)[:, None] >> np.arange(n)) & 1).astype(int)
energy = np.array([1.,0.,0.,1.])
gamma = np.pi/2  # γ(감마), π(파이)
after_cost = np.exp(-1j*gamma*energy)/2
print('비용 회로 후 진폭:',np.round(after_cost,6))
print('확률:',np.abs(after_cost)**2)
assert np.allclose(np.abs(after_cost)**2,[0.25]*4)

비용 회로 후 진폭: [0. -0.5j 0.5+0.j  0.5+0.j  0. -0.5j]
확률: [0.25 0.25 0.25 0.25]


In [23]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
gamma = np.pi/2  # γ(감마): 비용 각도
qc = QuantumCircuit(2)
qc.h([0,1])
qc.global_phase -= gamma/2
qc.rzz(gamma,0,1)
state = Statevector.from_instruction(qc)
print('비용 회로 후 진폭:',np.round(state.data,6))
print('확률:',state.probabilities())
assert np.allclose(state.data,[-0.5j,0.5,0.5,-0.5j])

비용 회로 후 진폭: [-0. -0.5j  0.5-0.j   0.5-0.j  -0. -0.5j]
확률: [0.25 0.25 0.25 0.25]


In [24]:
# 믹서와 간섭을 계산하기
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
gamma = np.pi/2  # γ(감마): 비용 각도
qc = QuantumCircuit(2)
qc.h([0,1])
qc.global_phase -= gamma/2
qc.rzz(gamma,0,1)
beta = -np.pi/8  # β(베타): 믹서 각도
qc.rx(2*beta,0)
qc.rx(2*beta,1)
state = Statevector.from_instruction(qc)
print('믹서 후 진폭:',np.round(state.data,6))
print('믹서 후 확률:',state.probabilities())
assert np.allclose(state.probabilities(),[0,0.5,0.5,0])

믹서 후 진폭: [0.      +0.j 0.707107-0.j 0.707107-0.j 0.      +0.j]
믹서 후 확률: [3.4666739e-33 5.0000000e-01 5.0000000e-01 3.4666739e-33]


In [25]:
import numpy as np
beta = -np.pi/8  # β(베타), π(파이)
a = -0.5j
b = 0.5+0j
a_final = a*np.cos(2*beta)-1j*b*np.sin(2*beta)
b_final = b*np.cos(2*beta)-1j*a*np.sin(2*beta)
print('00·11의 진폭:',a_final)
print('01·10의 진폭:',b_final)
print('상태별 확률:',np.abs([a_final,b_final,b_final,a_final])**2)
assert np.isclose(a_final,0) and np.isclose(abs(b_final)**2,0.5)

00·11의 진폭: -5.551115123125783e-17j
01·10의 진폭: (0.7071067811865475+0j)
상태별 확률: [3.08148791e-33 5.00000000e-01 5.00000000e-01 3.08148791e-33]


In [26]:
# 층수와 각도 최적화
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
energy = np.array([1.,0.,0.,1.])
scale = 1.0
p = 2
theta = np.array([np.pi/2,-np.pi/8,0.,0.])  # θ(세타): γ(감마), β(베타) 순서
def circuit(theta):
    qc = QuantumCircuit(2)
    qc.h([0,1])
    for layer in range(p):
        gamma = theta[2*layer]    # γ(감마)
        beta = theta[2*layer+1]   # β(베타)
        qc.global_phase -= gamma/(2*scale)
        qc.rzz(gamma/scale,0,1)
        qc.rx(2*beta,0)
        qc.rx(2*beta,1)
    return qc
qc = circuit(theta)
probability = Statevector.from_instruction(qc).probabilities()
print('층수:',p,'각도 수:',len(theta))
print('게이트 개수:',dict(qc.count_ops()))
print('확률:',probability)
assert len(theta)==2*p and np.isclose(probability.sum(),1)

층수: 2 각도 수: 4
게이트 개수: {'rx': 4, 'h': 2, 'rzz': 2}
확률: [3.4666739e-33 5.0000000e-01 5.0000000e-01 3.4666739e-33]


In [27]:
# 기대 에너지와 고전 최적화
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
energy = np.array([1.,0.,0.,1.])
scale = 1.0
p = 2
theta = np.array([np.pi/2,-np.pi/8,0.,0.])  # θ(세타): γ(감마), β(베타) 순서
def circuit(theta):
    qc = QuantumCircuit(2)
    qc.h([0,1])
    for layer in range(p):
        gamma = theta[2*layer]    # γ(감마)
        beta = theta[2*layer+1]   # β(베타)
        qc.global_phase -= gamma/(2*scale)
        qc.rzz(gamma/scale,0,1)
        qc.rx(2*beta,0)
        qc.rx(2*beta,1)
    return qc
from scipy.optimize import minimize
def objective(theta):
    probability = Statevector.from_instruction(circuit(theta)).probabilities()
    return float(probability@energy/scale)
initial = np.array([0.8,0.3,0.8,0.3])
initial_value = objective(initial)
result = minimize(objective,initial,method='COBYLA',
                  options={'maxiter':160,'rhobeg':0.5,'tol':1e-5})
probability = Statevector.from_instruction(circuit(result.x)).probabilities()
value = float(probability@energy/scale)
print('초기 기대 비용:',initial_value)
print('탐색 후 기대 비용:',value)
print('각도:',result.x)
print('종료 사유:',result.message)
print('평가 횟수:',result.nfev)
assert np.isclose(probability.sum(),1) and np.isfinite(value)

초기 기대 비용: 0.9384442191587989
탐색 후 기대 비용: 4.5082169155426254e-11
각도: [1.78311857 0.91793449 0.24487019 0.2634422 ]
종료 사유: Return from COBYLA because the trust region radius reaches its lower bound.
평가 횟수: 101


In [28]:
# 측정, 관측 비율과 최종 선택
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
gamma = np.pi/2  # γ(감마): 비용 각도
qc = QuantumCircuit(2)
qc.h([0,1])
qc.global_phase -= gamma/2
qc.rzz(gamma,0,1)
beta = -np.pi/8  # β(베타): 믹서 각도
qc.rx(2*beta,0)
qc.rx(2*beta,1)
state = Statevector.from_instruction(qc)
probability = state.probabilities()
energy = np.array([1.,0.,0.,1.])
valid = energy==0
shots = 4096
state.seed(42)
counts = state.sample_counts(shots)
valid_probability = probability[valid].sum()
valid_count = sum(count for label,count in counts.items() if valid[int(label,2)])
observed = [int(label,2) for label in counts if valid[int(label,2)]]
best = min(observed,key=lambda i:energy[i])
print('관측 횟수:',counts)
print('유효해 확률:',valid_probability)
print('유효해 관측 비율:',valid_count/shots)
print('관측된 유효 후보 중 최소 비용:',energy[best])
assert sum(counts.values())==shots and np.isclose(valid_probability,1)

관측 횟수: {np.str_('01'): np.int64(2074), np.str_('10'): np.int64(2022)}
유효해 확률: 0.9999999999999996
유효해 관측 비율: 1.0
관측된 유효 후보 중 최소 비용: 0.0


In [29]:
# 전체 코드의 계수와 회로를 읽는 순서
import numpy as np
constant = 0.0
a = np.zeros(2)
b = {}
penalty = 100.0
target = 1
indices = [0,1]
constant += penalty*target**2
for i in indices:
    a[i] += penalty*(1-2*target)
for position,i in enumerate(indices):
    for j in indices[position+1:]:
        key = tuple(sorted((i,j)))
        b[key] = b.get(key,0.0)+2*penalty
print('상수:',constant,'선형 계수:',a,'이차 계수:',b)
assert constant==100 and np.allclose(a,[-100,-100]) and b[(0,1)]==200

상수: 100.0 선형 계수: [-100. -100.] 이차 계수: {(0, 1): 200.0}


In [30]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
qc = QuantumCircuit(2)
qc.h([0,1])
gamma = 0.7  # γ(감마): 비용 각도
scale = 1.0
h = np.array([0.2,-0.3])
zz = {(0,1):0.5}
for i,value in enumerate(h):
    qc.rz(2*gamma*value/scale,i)
for (i,j),value in zz.items():
    qc.rzz(2*gamma*value/scale,i,j)
state = Statevector.from_instruction(qc)
print('게이트 개수:',dict(qc.count_ops()))
print('비용 회로 후 진폭:',np.round(state.data,6))
print('확률:',state.probabilities())
assert np.allclose(state.probabilities(),[0.25]*4)

게이트 개수: {'h': 2, 'rz': 2, 'rzz': 1}
비용 회로 후 진폭: [0.480528-0.138178j 0.382421+0.322109j 0.5     +0.j
 0.456544-0.20388j ]
확률: [0.25 0.25 0.25 0.25]
